In [0]:
from pyspark.sql.functions import *


In [0]:
base_path = '/Volumes/workspace/default/orders/'

channels_df= spark.read.csv(
    base_path + 'dim_channels.csv',
    header = True,
    inferSchema = True
)

couriers_df = spark.read.csv(
    base_path + 'dim_couriers.csv',
    header = True,
    inferSchema = True
)
customers_df = spark.read.csv(
    base_path + 'dim_customers.csv',
    header = True,
    inferSchema = True
)

products_df = spark.read.csv(
    base_path + 'dim_products.csv',
    header = True,
    inferSchema= True
)

warehouses_df = spark.read.csv(
    base_path + 'dim_warehouses.csv',
    header = True,
    inferSchema = True
)

orders_df = spark.read.csv(
    base_path + 'fact_orders.csv',
    header = True,
    inferSchema = True
)

order_item_df = spark.read.csv(
    base_path + 'fact_order_items.csv',
    header = True, 
    inferSchema= True
)

payments_df = spark.read.csv(
    base_path + 'fact_payments.csv',
    header = True,
    inferSchema = True
)

returns_df = spark.read.csv(
    base_path + 'fact_returns.csv',
    header = True,
    inferSchema = True
)

shipments_df = spark.read.csv(
    base_path + 'fact_shipments.csv',
    header = True,
    inferSchema = True
)

web_sessions_df = spark.read.csv(
    base_path + 'fact_web_sessions.csv',
    header = True,
    inferSchema = True
)

In [0]:
channels_df.show(5)
couriers_df.show(5)
customers_df.show(5)
products_df.show(5)
warehouses_df.show(5)

orders_df.show(5)
order_item_df.show(5)
payments_df.show(5)
returns_df.show(5)
shipments_df.show(5)
web_sessions_df.show(5)


In [0]:
channels_df.printSchema()
print('channels: ',channels_df.count())
couriers_df.printSchema()
print('couriers: ',couriers_df.count())
customers_df.printSchema()
print('customers: ',customers_df.count())
products_df.printSchema()
print('products: ',products_df.count())
warehouses_df.printSchema()
print('warehouses: ',warehouses_df.count())

orders_df.printSchema()
print('orders: ',orders_df.count())
order_item_df.printSchema()
print('order_items: ',order_item_df.count())
payments_df.printSchema()
print('payments: ', payments_df.count())
web_sessions_df.printSchema()
print('web_sessions: ',web_sessions_df.count())
returns_df.printSchema()
print('returns: ',returns_df.count())
shipments_df.printSchema()
print('shipments: ',shipments_df.count())

In [0]:
customers_cleaned_df= customers_df.fillna({'city':'Unknown'})
customers_cleaned_df= customers_cleaned_df.withColumn(
    'customer_name', 
    initcap(trim(col('customer_name')))
    )

customers_cleaned_df.show(5)

In [0]:
payments_cleaned_df = payments_df.fillna({'amount_paid': 0})

payments_cleaned_df= payments_cleaned_df.withColumn(
    'payment_gap',
    col('amount_due') - col('amount_paid')
)

payments_cleaned_df = payments_cleaned_df.withColumn(
    'payment_gap_status',
    when(col("payment_gap")==0,'Fully Paid')
    .when(col("payment_gap")>0, 'Payment Gap')
    .otherwise('Overpaid/ check')
    )
payments_cleaned_df.show(5)

In [0]:
orders_cleaned_df = orders_df.withColumn(
    'order_timestamp',
    to_timestamp(col('order_datetime'))
    )

orders_cleaned_df = orders_cleaned_df.withColumn(
    'order_date',
    to_date(col('order_datetime'))
)

orders_cleaned_df = orders_cleaned_df.withColumn(
    'order_year',
    year(col('order_timestamp'))
)

orders_cleaned_df = orders_cleaned_df.withColumn(
    'order_month',
    month(col('order_timestamp'))
)



orders_cleaned_df.show(5)

In [0]:
order_revenue_df = order_item_df.groupBy('order_id').agg(
    round(sum('net_revenue'),2).alias('order_revenue'),
    sum('quantity').alias('total_items')
)

order_revenue_df.show(10)


In [0]:
order_profile_df = orders_cleaned_df.join(
    order_revenue_df,
    on = 'order_id',
    how = 'left'
)

order_profile_df.show(5)